In [ ]:
# NOTEBOOK NAME
# RadarPlots.ipynb (SINGLE NOTEBOOK
# NOTEBOOK NAME

# OPENING IMPORTS

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions/')
from CustomFunctions1 import *
from RadarPlotsCustomFunctions import *

import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# mapping things
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

import matplotlib.colors as mcolors
import matplotlib.cm as cm

# for adding lat/lon gridlines on plots
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

from matplotlib.patches import Circle # for radar range ring circles on the map
from matplotlib.lines import Line2D # for plotting stars on the map legend


# # for projecting radar coordinates to lat and lon
# from pyproj import Geod

# import matplotlib.dates as mdates # for putting dates on the time axis of plots

from matplotlib.collections import LineCollection # plotting many lines at once

# sys.path.append('/home/563/sg3241/Notebooks/CustomFeatureTracking')
# from CustomTracking import *

# from scipy.stats import gaussian_kde # for plotting kernel density plots



In [ ]:
# HORIZONTAL CROSS SECTION PLOTTING (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)
# (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# retrieve pieces of the date and construct one without dashes
YearStr = PlotDate[0:4]
MonthStr = PlotDate[5:7]
DayStr = PlotDate[8:10]
FileDateStr = YearStr + MonthStr + DayStr

# retrieve radar name (and if the longitude is stored incorectly, get the shift amount [degrees]) (I am talking about Townsville (106)) 
RadarSiteName, LonShift = GrabRadarInfo(RadarIDno)

# Get variable info
VarInfo = GrabVarInfo(PlotVariable, ValueOrTextureOrCount, VarianceGridSize)

# Access individual fields
VarName           = VarInfo['VarName']
VarNameLong       = VarInfo['VarNameLong']
VarMinVal         = VarInfo['VarMinVal']
VarMaxVal         = VarInfo['VarMaxVal']
VarUnit           = VarInfo['VarUnit']
VarFillValue      = VarInfo['VarFillValue']
VarColourBar      = VarInfo['VarColourBar']
VarColourBar_min  = VarInfo['VarColourBar_min']
VarColourBar_max  = VarInfo['VarColourBar_max']
VarColourBar_norm = VarInfo['VarColourBar_norm']
VarTickSpacing    = VarInfo['VarTickSpacing']


# FEATURE DATA LOADING
if IncludeFeatureDots or IncludeFeatureTracks:
    FeatureStoragePath = f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/{RadarIDno}/V5/{FileDateStr}/{RadarIDno}_{FileDateStr}FeatureStats.nc'
    FeatureXR = xr.open_dataset(FeatureStoragePath)
    
    # Create a new time dimension for time of day, instead of just time in a given feature's life
    # Generate 288 x 5-minute timestamps to assign as coordinate along the 'times' dimension
    RadarFileDatePD = pd.Timestamp(PlotDate).date()
    FrameTimes = pd.date_range(start=pd.Timestamp(RadarFileDatePD), periods=288, freq='5min')
    FeatureXR = FeatureXR.assign_coords(FrameTimes=xr.DataArray(FrameTimes, dims='FrameTimes'))
    
    # add variables that rearrange times by time of day rather than time in a given feature's life
    FeatureXR = AddFrameTimeVars(FeatureXR)


# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])
# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin
for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)

    if (DataSourceType=='Grid'):
            NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/NetCDFs/CompressedRadarGrids/' + str(RadarIDno) + '/' + YearStr + '/' + MonthStr + '/' + DayStr + '/'
            NetCDFstorageFile = str(RadarIDno) + '_' + FileDateStr + '_' + RadarFileTime + '.nc'
            NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile
    elif (DataSourceType=='PPI'):
            NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + str(RadarIDno) + '/' + str(RadarIDno) + '_' + FileDateStr + '_ppi' + '/'
            NetCDFstorageFile = str(RadarIDno) + '_' + FileDateStr + '_' + RadarFileTime + '_ppi.nc'
            NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile
    else:
        print("Source Type must be exactly 'Grid' or 'PPI'")


    
    # try to load in the netcdf file and if it doesn't work, just keep going through the loop
    try:
        RadarXR = xr.open_dataset(NetCDFstoragePath)
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue


    if (PlotType == 'Horizontal'):
        # create the figure to be plotted on with a map projection
        fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    
        # set the plot background to black and text to white if the user wants it
        if (PlotColourStyle=='Dark'):
            fig.patch.set_facecolor('#0a0a0a')  
            ax.set_facecolor('#0a0a0a')
            StandOutColour = 'white'
        elif (PlotColourStyle=='Light'):
            StandOutColour = 'black'
        else:
            print("PlotColourStyle must be exactly 'Light' or 'Dark'")
    
        # add a background topo map to the plot
        ElevationModelPath = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
        # Extract plot bounds from radar grid
        if (DataSourceType == 'Grid'):
            LonMin, LonMax = float(RadarXR.lon.min()) + LonShift, float(RadarXR.lon.max()) + LonShift
            LatMin, LatMax = float(RadarXR.lat.min()), float(RadarXR.lat.max())
        elif (DataSourceType == 'PPI'):
    
            # calculate how many degrees this range is (different for latitude vs longitude)
            RadiusEarth = 6371 # [km]
            kmPerDegLat = (RadiusEarth * 2 * np.pi) /360
            kmPerDegLon = kmPerDegLat * np.cos( np.deg2rad(RadarXR.latitude))
        
            # calculate the ground range when beam altitude reaches 20 km for a max ground radius of consideration 
            RangeOf20km = 20 / (np.tan(np.deg2rad(ElevationAngle)))
    
            # choose how far you want the extremes of the box in the plot from the radar
            BoxRange = np.minimum(150.0, RangeOf20km ) # [km] set that maximum ground range as a limit to the plot, or 150 km as a max value
            # box range in degrees instead of km
            BoxRangeDegLat = BoxRange / kmPerDegLat
            BoxRangeDegLon = BoxRange / kmPerDegLon
    
            # Extract plot bounds from distance from radar
            LonMin, LonMax = float(RadarXR.longitude) + LonShift - BoxRangeDegLon, float(RadarXR.longitude) + LonShift + BoxRangeDegLon
            LatMin, LatMax = float(RadarXR.latitude)   - BoxRangeDegLat,  float(RadarXR.latitude)  + BoxRangeDegLat
    
        else:
            print("DataSourceType must be exactly 'Grid' or 'PPI'")
        # call the "adding-the-map" function
        AddTopoShading(ax, LonMin, LonMax, LatMin, LatMax, ElevationModelPath, PlotColourStyle)
    
    
        # PLOT THE MAIN VARIABLE
        if (DataSourceType == 'Grid'):
            # index in the netcdf altitude variable for the altitude you want
            alti = np.where(RadarXR.z == Altitude) # this is a double nested array for some reason
            alti = alti[0][0] # take the index out of the double nested array
        
            # quit out if the altitude does not correspond to one in the netCDF file
            if ( np.size(alti) != 1): 
                raise ValueError( str(Altitude) + ' m is not a valid altitude in the data')
            
            PlottingArray = RadarXR[VarNameLong][0,alti,:,:]
            GridViewer = ax.pcolormesh(RadarXR.lon+ LonShift, RadarXR.lat, PlottingArray, 
                                       cmap=VarColourBar , norm=VarColourBar_norm, transform=ccrs.PlateCarree())
    
            # plot a star for the location of the radar on the map
            ax.plot(float(RadarXR.radar_longitude[0]) + LonShift, float(RadarXR.radar_latitude[0]),
                    marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=20)
            ax.plot(float(RadarXR.radar_longitude[0]) + LonShift, float(RadarXR.radar_latitude[0]),
                    marker='*', color='white', markersize=4, transform=ccrs.PlateCarree(), zorder=21)
    
            plt.title(VarName + ' for ' + RadarSiteName + ' Radar\nat ' + str(Altitude) + ' m Altitude\non ' + \
                PlotDate + ' at ' + \
                str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC', color = StandOutColour)
        
        elif (DataSourceType == 'PPI'):
            # ADD COORDINATES FOR EACH GATE SO THE DATA CAN BE PLOTTED
            RadarXR = AddGateCoords(RadarXR, LonShift)
            # select the "time coordinates" for the given elevation angle
            EleAngleMask = (RadarXR.elevation == ElevationAngle) # make a mask to only plot the one elevation angle
            PlotLats = RadarXR.gate_latitude[EleAngleMask]
            PlotLons = RadarXR.gate_longitude[EleAngleMask]
            PlotVar = RadarXR[VarNameLong][EleAngleMask]
    
            PlotVar = PlotVar.where(PlotVar != VarFillValue, np.nan) # get rid of the fill values in the variable you are plotting
            GridViewer = ax.pcolormesh( PlotLons, PlotLats, PlotVar, cmap=VarColourBar, norm=VarColourBar_norm, shading='auto', transform=ccrs.PlateCarree() )
    
            # plot a star for the location of the radar on the map
            ax.plot(float(RadarXR.longitude) + LonShift, float(RadarXR.latitude),
                    marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=20)
            ax.plot(float(RadarXR.longitude) + LonShift, float(RadarXR.latitude),
                    marker='*', color='white', markersize=4, transform=ccrs.PlateCarree(), zorder=21)
    
            plt.title(VarName + ' for ' + RadarSiteName + ' Radar\nat the ' + str(ElevationAngle) + '° Elevation Angle\non ' + \
                PlotDate + ' at ' + \
                str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC', color = StandOutColour)
            
        else:
            print("DataSourceType must be exactly 'Grid' or 'PPI'")
    
        # colour bar controls
        cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
        cbar.set_label(VarName + ' [' + VarUnit + ']', color = StandOutColour)
        cbar.ax.yaxis.set_tick_params(color = StandOutColour)
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color = StandOutColour)
        cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
        cbar.set_ticks(np.arange(VarMinVal, VarMaxVal+0.00001, VarTickSpacing))  # control tick spacing
    
        # ADD LAT/LON GRID LINES 
        # (set thickness of grid lines) 
        ThinLineThickness    = 0.2
        MediumLineThickness  = 0.3
        ThickLineThickness   = 0.6
        # (and what multiples of lat/lon have lines)
        ThinLineFrequency    = 0.1
        MediumLineFrequency  = 0.5
        ThickLineFrequency   = 1.0
        # alright, lets add those lines
        AddGridlines(ax, ThinLineThickness, MediumLineThickness, ThickLineThickness,
                         ThinLineFrequency,  MediumLineFrequency,  ThickLineFrequency, StandOutColour)
    
        # set the plot limits to the same as the BACKGROUND TOPO MAP
        plt.xlim([LonMin, LonMax])
        plt.ylim([LatMin, LatMax])

        # PLOT FEATURE DOTS IF THE USER ASKS FOR THEM
        if IncludeFeatureDots:
        
            # Derive the timestamp and find the matching FrameTimes index
            RadarFileTimeStamp = pd.Timestamp(f'{PlotDate}T{RadarFileTimePrint}').floor('5min')
            TimeIndex = pd.DatetimeIndex(FeatureXR['FrameTimes'].values).get_loc(RadarFileTimeStamp)
        
        
            # Extract values at this time index
            PlotLons = FeatureXR['meanlon_frametimes'].sel(FrameTimes=FeatureXR['FrameTimes'][TimeIndex]).values
            PlotLats = FeatureXR['meanlat_frametimes'].sel(FrameTimes=FeatureXR['FrameTimes'][TimeIndex]).values
            
            # Mask to only tracks that exist at this time (i.e. not NaN or -9999)
            ValidMask = ~np.isnan(PlotLons) & ~np.isnan(PlotLats)

            if (PlotColourStyle == 'Light'):
                DotsThemeColour = [0.8, 0.0, 0.0]
            elif(PlotColourStyle == 'Dark'):
                DotsThemeColour = [0.0, 0.5, 0.8]
            else:
                print("PlotColourStyle must be exactly 'Light' or 'Dark'")
            
            # colourful dots with white in the middle
            ax.scatter(
                PlotLons[ValidMask],
                PlotLats[ValidMask],
                s         = 6,
                color = DotsThemeColour,
                transform = ccrs.PlateCarree(),
                zorder    = 25,
            )
        
            ax.scatter(
                PlotLons[ValidMask],
                PlotLats[ValidMask],
                s         = 1,
                color     = [1,1,1],
                transform = ccrs.PlateCarree(),
                zorder    = 26,
            )

        if IncludeFeatureTracks:
            # NOW FOR THE TRAILING LINES
            # --- Separate tracks into categories ---
            # For each track, find the slice of frametimes up to and including TimeIndex
            TrackLons = FeatureXR['cell_meanlon_frametimes'].values  # shape: (tracks, FrameTimes)
            TrackLats = FeatureXR['cell_meanlat_frametimes'].values  # shape: (tracks, FrameTimes)
            
            Lines        = []  # for LineCollection
            SingleDots_x = []  # one-frame dead features
            SingleDots_y = []
            
            for i in range(TrackLons.shape[0]):
                
                # Get this track's data up to and including TimeIndex
                LonSlice = TrackLons[i, :TimeIndex + 1]
                LatSlice = TrackLats[i, :TimeIndex + 1]
            
                # Valid mask — non-NaN points up to present
                ValidMask = ~np.isnan(LonSlice) & ~np.isnan(LatSlice)
            
                # Skip features that do not exist yet
                if ValidMask.sum() == 0:
                    continue
            
                ValidLons = LonSlice[ValidMask]
                ValidLats = LatSlice[ValidMask]
            
                if ValidLons.shape[0] == 1:
                    # Single point — add to dot list
                    SingleDots_x.append(ValidLons[0])
                    SingleDots_y.append(ValidLats[0])
                else:
                    # Build coordinate pairs for LineCollection
                    Points = np.array([ValidLons, ValidLats]).T  # shape: (n, 2)
                    Segments = np.stack([Points[:-1], Points[1:]], axis=1)  # shape: (n-1, 2, 2)
                    Lines.append(Segments)

            if (PlotColourStyle == 'Light'):
                TracksThemeColour = [0.9, 0.2, 0.0]
            elif(PlotColourStyle == 'Dark'):
                TracksThemeColour = [0.0, 0.7, 0.9]
            else:
                print("PlotColourStyle must be exactly 'Light' or 'Dark'")
            
            # --- Draw lines ---
            if len(Lines) > 0:
                AllSegments = np.concatenate(Lines, axis=0)
                LC = LineCollection(
                    AllSegments,
                    color      = TracksThemeColour,
                    linewidths = 0.3,
                    alpha      = 0.3,
                    transform  = ccrs.PlateCarree(),
                    zorder     = 24,
                )
                ax.add_collection(LC)
            
            # --- Draw single-point dots ---
            if len(SingleDots_x) > 0:
                ax.scatter(
                    SingleDots_x,
                    SingleDots_y,
                    s         = 1,
                    alpha      = 0.3,
                    color     = TracksThemeColour,
                    transform = ccrs.PlateCarree(),
                    zorder    = 24.5,
                )

        
    elif(PlotType == 'Vertical'):
        # CHOOSE YOUR ALTITUDE RANGE
        MinHeight = 0   # [km] minimum height in plot

        # Load in a DEM (update path and variable name as needed)
        DEMpath = '/home/563/sg3241/QueenslandElevationGEBCO.nc'  # <- your DEM file
        DEMdata = xr.open_dataset(DEMpath)
        DEMelev = DEMdata['elevation']  # adjust if your var has a different name

        # sounding data follow-on and loading
        SoundingFolder = '/home/563/sg3241/TownsvilleSoundings/' 
        SoundingFileName = FileDateStr + f'{UpperAirHour:02d}' + '-' + str(UpperAirSiteIDno) + '.csv'
        SoundingPath = SoundingFolder + SoundingFileName
        SoundingData = pd.read_csv(SoundingPath)

                # RADAR BEAM CENTRE LINES SECTION
        # Elevation angles of the radar beams [degrees]
        elevation_angles = np.array([0.5, 0.8, 1.4, 2.4, 3.5, 4.7, 6.0, 7.8, 10, 13, 17, 23, 32])
    
        # Calculate beam centre heights using curved Earth model
        x_vals   = RadarXR.x.values * 0.001  # east-west distance from radar [km]
        y_vals   = RadarXR.y.values * 0.001  # east-west distance from radar [km]
        z_vals   = RadarXR.z.values * 0.001  # height above surface [km]

        # Coordinates in km (range from radar)
        xValsKM = np.array(RadarXR.x) * 0.001  # x coordinates in km
        yValsKM = np.array(RadarXR.y) * 0.001  # y coordinates in km
        
        # farthest south and north x-values
        xMinSliceKM = int(np.min(xValsKM))
        xMaxSliceKM = int(np.max(xValsKM))

        # farthest east and west y-values
        yMinSliceKM = int(np.min(yValsKM))
        yMaxSliceKM = int(np.max(yValsKM))
    
        # find the vertical cross section index in the coordinates
        if (SliceDirection == 'North'):
            Slicei = np.where(yValsKM == SliceDistance)[0][0]       # index in the y-coordinates where that north-south km value lives      
            horz_vals = np.sqrt(x_vals**2 + SliceDistance**2)  # Horizontal ground distance from radar at each x point along the slice
            AlongDirection = 'East'
            CoordSlice = RadarXR['lon'][Slicei, :].values
            PlottingArray = RadarXR[VarNameLong][0,:,Slicei,:]
        elif(SliceDirection == 'South'):
            Slicei = np.where(yValsKM == SliceDistance * -1)[0][0]  # index in the y-coordinates where that north-south km value lives (negative for south)
            horz_vals = np.sqrt(x_vals**2 + SliceDistance**2)  # Horizontal ground distance from radar at each x point along the slice
            AlongDirection = 'East'
            CoordSlice = RadarXR['lon'][Slicei, :].values
            PlottingArray = RadarXR[VarNameLong][0,:,Slicei,:]
        elif(SliceDirection == 'East'):
            Slicei = np.where(xValsKM == SliceDistance)[0][0]       # index in the x-coordinates where that north-south km value lives
            horz_vals = np.sqrt(y_vals**2 + SliceDistance**2)  # Horizontal ground distance from radar at each x point along the slice
            AlongDirection = 'South'
            CoordSlice = RadarXR['lat'][:, Slicei].values
            PlottingArray = RadarXR[VarNameLong][0,:,:,Slicei]
        elif(SliceDirection =='West'):
            Slicei = np.where(xValsKM == SliceDistance* -1)[0][0]   # index in the x-coordinates where that north-south km value lives (negative for west)
            horz_vals = np.sqrt(y_vals**2 + SliceDistance**2)  # Horizontal ground distance from radar at each x point along the slice
            AlongDirection = 'South'
            CoordSlice = RadarXR['lat'][:, Slicei].values
            PlottingArray = RadarXR[VarNameLong][0,:,:,Slicei]
        else:
            print("SliceDirection must be exactly 'North', 'South', 'East' or 'West'")

        # beam_altitude_curves shape: (n_angles, nx)
        beam_altitude_curves = np.array([ beam_height_curved(horz_vals, elev_angle) for elev_angle in elevation_angles ])

        # EXPERIMENTAL TOPOGRAPHY SECTION
        # EXPERIMENTAL TOPOGRAPHY SECTION
        # EXPERIMENTAL TOPOGRAPHY SECTION
    
        # lat/lon fields in RadarXR: dimensions (y, x)
        # Slice at the chosen north–south index EWslicei along x
        lat_slice = RadarXR['lat'][Slicei, :].values  # shape (nx,)
        lon_slice = RadarXR['lon'][Slicei, :].values  # shape (nx,)
    
        # Build DataArrays for interpolation
        LONdata = xr.DataArray(lon_slice, dims=('x',))
        LATdata = xr.DataArray(lat_slice, dims=('x',))
    
        # Interpolate DEM to the cross-section line
        dem_slice = DEMelev.interp(lon=LONdata, lat=LATdata)
    
        # Elevation in metres along the line
        TerrainSlice = dem_slice.values
    
        # For plotting in km, and do not let negative (ocean) go below 0
        TerrainSliceKM = np.maximum(TerrainSlice, 0.0) * 0.001
    
        Xkm = RadarXR.x * 0.001  # east–west distance [km]
    
        # EXPERIMENTAL TOPOGRAPHY SECTION END
        # EXPERIMENTAL TOPOGRAPHY SECTION END
        # EXPERIMENTAL TOPOGRAPHY SECTION END
    
        # MAIN DATA PLOTTING
        fig, ax = plt.subplots(figsize=(25, 3))
        
        GridViewer = pcolormeshC(CoordSlice, RadarXR.z * 0.001, PlottingArray, ax=ax, 
                                 cmap=VarColourBar, norm=VarColourBar_norm)
        
        # Plot radar beam centre lines as curves across the slice
        for i, elev_angle in enumerate(elevation_angles):
            beam_curve = beam_altitude_curves[i, :]  # altitude [km] at each x point along the slice
        
            # Only plot if any part of the beam is within the plot range
            if np.any(beam_curve <= MaxPlotAltitude):
                ax.plot(CoordSlice, beam_curve, color=[0.5, 0.5, 1.0], linewidth=2.0,
                        alpha=0.15, linestyle='-', zorder=2)
        
                # Find a sensible x position for the label — use the rightmost point still in range
                in_range_mask = beam_curve <= MaxPlotAltitude
                label_idx = np.where(in_range_mask)[0][-1]  # rightmost in-range index
        
                ax.text(CoordSlice[label_idx], beam_curve[label_idx], f'{elev_angle}°',
                        fontsize=7, va='bottom', ha='center',
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))
    
        # Plot the line showing ground/topography
        ax.plot(CoordSlice, TerrainSliceKM, color=[0.3, 0.3, 0.3], linewidth=0.8, zorder=5)
        ax.fill_between(CoordSlice, 0, TerrainSliceKM, color=[0.3, 0.3, 0.3], zorder=4)
        
        cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']', pad=0.01)
    
        cbar.ax.set_ylim(VarMinVal, VarMaxVal)
        cbar.set_ticks(np.arange(VarMinVal, VarMaxVal+0.0001, VarTickSpacing))
        
        # ADD A FUNCTION THAT CAN FIND THE TROPOPAUSE !!!
        # ADD A FUNCTION THAT CAN FIND THE TROPOPAUSE !!!
        # ADD A FUNCTION THAT CAN FIND THE TROPOPAUSE !!!
        # TropopauseAlt = 16.8
        # # plot the tropopause
        # ax.axhline(y=TropopauseAlt, color=[0.6, 0.0, 0.0], linewidth=0.6, label='Tropopause')
        
        # plot the bottom and top of the Dentritic Growth Zone Temperature Range
        ax.axhline(y=temp_crossing_altitude(SoundingData, -10), color=[0.0, 0.8, 0.8], linewidth=0.6, alpha=0.4, label='Dentritic Growth Zone Temperature Range (-10 to -20°C)')
        ax.axhline(y=temp_crossing_altitude(SoundingData, -20), color=[0.0, 0.8, 0.8], linewidth=0.6, alpha=0.4)
        
        # plot the bottom and top of the Hallett-Mossop Temperature Range
        ax.axhline(y=temp_crossing_altitude(SoundingData, -3), color=[0.8, 0.4, 0.0], linewidth=0.6, alpha=0.4, label='Hallett-Mossop Temperature Range (-8 to -3°C)')
        ax.axhline(y=temp_crossing_altitude(SoundingData, -8), color=[0.8, 0.4, 0.0], linewidth=0.6, alpha=0.4)
        
        # plot the freezing level
        ax.axhline(y=temp_crossing_altitude(SoundingData, 0), color='blue', linewidth=0.6, alpha=0.4, label='Freezing Level (0°C)')
        

        # approximate latitude where the slice is taken (may vary a bit since the radar grid isn't perfect)
        if (SliceDirection == 'North') or (SliceDirection == 'North'):

            ApproxLat = round(float(lat_slice[150]), 2)
            # change it to positive and write 'north' or 'south'
            if (ApproxLat < 0):
                ApproxLat = ApproxLat * -1
                ApproxLatDir = 'S'
            else:
                ApproxLatDir = 'N'
                
        elif (SliceDirection == 'East') or (SliceDirection == 'West'):
            
            ApproxLat = round(float(lon_slice[150]), 2)
            # change it to positive and write 'east' or 'west'
            if (ApproxLat < 0):
                ApproxLat = ApproxLat * -1
                ApproxLatDir = 'W'
            else:
                ApproxLatDir = 'E'
        else:
            print("SliceDirection must be exactly 'North', 'South', 'East' or 'West'")



            
        
        ax.set_xlabel('Longitude [Degrees ' + AlongDirection + ']')
        ax.set_ylabel('Altitude [km]')
        plt.title(VarName + ' Cross Section for ' + RadarSiteName + ' Radar\n For Slice Taken ' + \
              str(SliceDistance) + ' km ' + SliceDirection + ' of the Radar (~' + str(ApproxLat) + '° ' + ApproxLatDir + ')\n' + \
              PlotDate + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC', pad=12)

        # add a grid with major, medium, and minor thicknesses
        ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.1))
        ax.xaxis.set_major_locator(mticker.MultipleLocator(0.5))
        
        ax.grid(which='minor', linewidth=0.3, alpha=0.2)
        ax.grid(which='major', linewidth=0.5, alpha=0.4)
        
        # thick lines every degree — drawn manually as axvlines
        for lon_tick in np.arange(np.floor(np.min(CoordSlice)), np.ceil(np.max(CoordSlice)) + 1, 1.0):
            ax.axvline(x=lon_tick, linewidth=0.9, alpha=0.5, color='grey', zorder=1)

        ax.set_xticks(np.arange(np.floor(np.min(CoordSlice)), np.ceil(np.max(CoordSlice)) + 0.5, 0.5))

        ax.yaxis.set_minor_locator(mticker.MultipleLocator(1))
        ax.yaxis.set_major_locator(mticker.MultipleLocator(2))

        for alt_tick in np.arange(0, MaxPlotAltitude + 1, 5):
            ax.axhline(y=alt_tick, linewidth=0.9, alpha=0.5, color='grey', zorder=1)

        ax.grid(which='minor', linewidth=0.3, alpha=0.3)
        ax.grid(which='major', linewidth=0.5, alpha=0.6)


        plt.xlim([np.min(CoordSlice), np.max(CoordSlice)])
        plt.ylim([0, MaxPlotAltitude])

        ax.legend(loc='upper left', fontsize='small', markerscale=0.7)

    
        
    else:
        print("PlotType must be exactly 'Horizontal' or 'Vertical'")
            


    # SAVE THE PLOT IF THE USER ASKED FOR IT
    if SavePlot:
        SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + str(RadarIDno) + '/' + FileDateStr + '/' + VarName + '/'

        SaveFile   = str(RadarIDno) + '_' + FileDateStr + '_' + RadarFileTime + '_' + VarName + '_' + PlotType + str(Altitude) + 'm.png'
        
        SavePath = SaveFolder + SaveFile
        
        if not Path(SaveFolder).exists():
            print('doing')
            Path(SaveFolder).mkdir(parents=True, exist_ok=True)
        
        plt.savefig(SavePath, bbox_inches='tight', facecolor='k', dpi = SaveResolution)

    # DISPLAY THE PLOT ONLY IF THE USER ASKED FOR IT
    if  not(DisplayPlot):
        plt.close()
    else:
        plt.show()



# PlotType               = 'Horizontal' # ['Horizontal' or 'Vertical' (string)]                                # Is this going to be a top-down (horizontal) looking-plot or a vertical slice plot?
# DataSourceType         = 'Grid'       # ['Grid' or 'PPI' (string)]                                 # Do you want to plot cartesian gridded data or the original spherical coordinate (PPI) data?
# QualityControl         = 'None'       # ['None' or 'Option1' or 'Option2' or 'Option3' (string)]   # Which quality control do you want to apply to the data you are looking at?

# # OPTIONS FOR CAPPIS
# Altitude               = 2000         # [m (int)] *must be divisible by 500 m*                     # At which altitude (above the radar???) would you like to take a horizontal slice?
# IncludeFeatureDots     = True        # ['True' or 'False' (boolean)]                              # Do you want to put dots at the centre of each tracked feature?
# IncludeFeatureTracks   = True        # ['True' or 'False' (boolean)]                              # Do you want to have lines showing where all of the features have travelled on this UTC day?
# PlotColourStyle        = 'Light'      # ['Light' or 'Dark' (string)]                                # Do you want a light or dark background for your plots?

# # OPTIONS FOR TOP-DOWN (HORIZONTAL) PPI PLOTS
# ElevationAngle         = 0.8          # [degrees] [see Elevation Angle Catalogue below (float)]    # Which Elevation angle do you want to plot data for?

# # FOR VERTICAL PLOTS
# SliceDirection         = 'South'      # ['North' or 'South' or 'East' or 'West' (string)]          # Which direction from the radar do you want the slice taken (slice will be perpendicular to this direction)?
# SliceDistance          = 25           # [km (int)]                                                 # How far from the radar you want to take the slice taken?
# MaxPlotAltitude        = 12           # [km (float)]                                               # how far up in altitude do you want this plot to go?  

In [ ]:
RadarXR

In [ ]:
# "NAMELIST BLOCK" #
# "NAMELIST BLOCK" #
# "NAMELIST BLOCK" #

# PLOT VERSION
PlotType               = 'Horizontal' # ['Horizontal' or 'Vertical' (string)]                                # Is this going to be a top-down (horizontal) looking-plot or a vertical slice plot?
DataSourceType         = 'PPI'       # ['Grid' or 'PPI' (string)]                                 # Do you want to plot cartesian gridded data or the original spherical coordinate (PPI) data?
QualityControl         = 'None'       # ['None' or 'Option1' or 'Option2' or 'Option3' (string)]   # Which quality control do you want to apply to the data you are looking at?
# NOTE THAT GRIDDED DATA IS ONLY FOR HORIZONTAL PLOTS FOR NOW!!!

# PLOT OPTIONS
DisplayPlot            = True         # ['True' or 'False' (boolean)]                              # Do you want to see the plot appear in this notebook when you run it?
SavePlot               = False        # ['True' or 'False' (boolean)]                              # Do you want to save the plot to a folder here on GADI?
SaveResolution         = 300          # [pixels per inch]                                          # How high of resolution would you want to save this plot?

# WHEN AND WHERE ARE YOU PLOTTING?
RadarIDno              = 41           # [see Radar Number Catalogue below (int)]                   # Which radar site are you plotting data for?
PlotDate               = '2024-03-09' # ['YYYY-MM-DD' (string)]                                    # Which (UTC) date are you plotting for?
LoopStartTime          = '15:00'      # ['hh:mm' (string)]                                         # *must be divisible by 5 min* Over what time span on this UTC day do you want to make plots for?
LoopEndTime            = '15:15'      # *use same time twice for single frame* 

# WHAT VARIABLE ARE YOU PLOTTING?
PlotVariable           = 'Z'          # [see Variable Catalogue below (string)]                    # Which variable do you want to plot?
ValueOrTextureOrCount  = 'Value'      # ['Value' or 'Texture' or 'Count' (string)]                 # Do you want to plot the variable itself, its texture (variance over a surrounding grid) or count of valid values in a grid surrounding each point?
VarianceGridSize       = '3x3'        # ['3x3' or '5x5' (string)]                                  # How large would you want this surrounding grid to be?



# OPTIONS FOR CAPPIS
Altitude               = 2000         # [m (int)] *must be divisible by 500 m*                     # At which altitude (above the radar???) would you like to take a horizontal slice?
IncludeFeatureDots     = True        # ['True' or 'False' (boolean)]                              # Do you want to put dots at the centre of each tracked feature?
IncludeFeatureTracks   = True        # ['True' or 'False' (boolean)]                              # Do you want to have lines showing where all of the features have travelled on this UTC day?
PlotColourStyle        = 'Dark'      # ['Light' or 'Dark' (string)]                                # Do you want a light or dark background for your plots?

# OPTIONS FOR TOP-DOWN (HORIZONTAL) PPI PLOTS
ElevationAngle         = 1.4          # [degrees] [see Elevation Angle Catalogue below (float)]    # Which Elevation angle do you want to plot data for?

# FOR VERTICAL PLOTS
SliceDirection         = 'South'      # ['North' or 'South' or 'East' or 'West' (string)]          # Which direction from the radar do you want the slice taken (slice will be perpendicular to this direction)?
SliceDistance          = 25           # [km (int)]                                                 # How far from the radar you want to take the slice taken?
MaxPlotAltitude        = 12           # [km (float)]                                               # how far up in altitude do you want this plot to go?
UpperAirSiteIDno       = 95282        # [ 95282 for Townsville or 94299 for Willis Island (int)]   # Which upper-air station would you like to plot temperatures from?
UpperAirHour           = 0            # [UTC hour of day (int)]                                    # from which UTC hour of the day would you like to use the sounding from (usually only 0 exists)    

# ??? PERHAPS WE CAN CHANGE THE VERTICAL PLOTS TO BE ABLE TO SHOW ANY CROSS-SECTION, NOT JUST PURELY EAST-WEST or NORTH-SOUTH LIKE WE DID FOR THE HOVMOLLER PLOTS ??? #

In [ ]:
# CHOICE CATALOGUES
# CHOICE CATALOGUES
# CHOICE CATALOGUES


# RADAR NUMBER CATALOGUE:

# Willis Island (not Dual-Pol): 99

# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)

#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald



# ELEVATION ANGLE CATALOGUE:

# MOST RADARS:
#  0.5   0.8   1.4   2.4
#  3.5   4.7   6.0   7.8
#   10    13    17    23   and   32

# ODD RADARS:
#  XXX   0.8   1.6   2.4
#  3.6   XXX   5.6   8.0
# 11.5   XXX    16    22   and   32



# VARIABLE CATALOGUE
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
# ...
# [V]         'corrected_velocity'

# VARIABLES NOT YET INCLUDED IN THE CODE
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
# ...
# [AzSh]      'azshear'

In [ ]:
# GIF MAKER
# FOR Horizontal Cross Sections

for RadarIDno in ['22']:

    SavedFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/' + \
                                                                                           VarName + '/RhoHVlim' + str(int(MinValidRhoHV*100)) + 'Compressed/'
                                                                                                      # RhoHV 0.85 becomes 85 in file name         
    # LOADING IMAGES
    files = sorted(os.listdir(SavedFolder)) # takes all of the files in the folder in the order they are named
    
    images = [Image.open(os.path.join(SavedFolder, f))
        for f in files
        if f.endswith(PlotType + str(Altitude) + 'm.png')]
    
    # DOWNSCALE IMAGES BEFORE MAKING GIF
    InverseScaleFactor = 3
    ScaleFactor = 1 / InverseScaleFactor
    # 0.5 = half resolution, 0.25 = quarter resolution etc.
    
    ResizedImages = []
    for img in images:
        NewWidth  = int(img.width  * ScaleFactor)
        NewHeight = int(img.height * ScaleFactor)
        ResizedImages.append(img.resize((NewWidth, NewHeight), Image.LANCZOS))
    
    GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/Horz/' + RadarIDno + '/' + RadarFileDate + '/'
    GIFsaveFile   = RadarIDno + '_' + RadarFileDate + '_' + VarName + '_RhoHVlim' + str(int(MinValidRhoHV*100)) + '_' + str(Altitude) + 'm.gif'
    
    GIFsavePath = GIFsaveFolder + GIFsaveFile
    
    # make a folder to store the new GIF in if one does not exist already
    if not Path(GIFsaveFolder).exists():
        Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)
    
    # Save as looping GIF
    ResizedImages[0].save(GIFsavePath, save_all=True, append_images=ResizedImages[1:], duration=200, loop=0)          
                                                                       # ms per frame       0 = loop forever
    print('Saved GIF for ' + RadarFileDate)